# LongFlow P0 — Stage 1: contrast-pair generation + activation capture

Runtime: **L4 GPU**. Spec: `docs/experiments/p0-steering.md` Stage 1; capture code: `src/steering/contrast_pairs.py`; scripts/lead-ins: `configs/p0_contrast.json`.

Plan: calibrate hook behavior on one tiny generation, then run 10 scripts × 2 axes × 2 poles × K=2 = **80 generations** (~1 sentence lead-in + 3 target sentences each), pool per-layer vectors, save `vectors.pt`. Budget: well under the sweep's 90-generation scale, expect 1–2 hrs wall-clock.

In [ ]:
!nvidia-smi -L
%cd /content
!git clone https://github.com/vibevoice-community/VibeVoice.git 2>/dev/null || true
%cd /content/VibeVoice
!git checkout -q 07cb79fea
!pip install -q -e .

LongFlow is a private repo — paste a GitHub personal access token (fine-grained, read-only on Josh-E-S/LongFlow) when prompted. Alternatively skip this cell and upload `contrast_pairs.py` + `p0_contrast.json` by hand to `/content/longflow_src/`.

In [ ]:
from getpass import getpass
token = getpass("GitHub token: ")
!git clone -q https://{token}@github.com/Josh-E-S/LongFlow.git /content/LongFlow
del token
import sys
sys.path.insert(0, "/content/LongFlow")
from src.steering.contrast_pairs import (
    LayerActivationRecorder, pool_record, save_records,
)
import json
CFG = json.load(open("/content/LongFlow/configs/p0_contrast.json"))
print(len(CFG["scripts"]), "scripts;", list(CFG["axes"]))

In [ ]:
import torch
from vibevoice.modular.modeling_vibevoice_inference import (
    VibeVoiceForConditionalGenerationInference,
)
from vibevoice.processor.vibevoice_processor import VibeVoiceProcessor

MODEL_ID = "microsoft/VibeVoice-1.5B"
model = VibeVoiceForConditionalGenerationInference.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map="cuda"
)
processor = VibeVoiceProcessor.from_pretrained(MODEL_ID)
model.eval()

# Stage-0 carry-over #1: record the exact weights revision
from huggingface_hub import snapshot_download
import re, os
snap = snapshot_download(MODEL_ID)
print("weights revision:", os.path.basename(os.path.realpath(snap)))

# Stage-0 carry-over #2: the actual solver step count
for attr in ("ddpm_inference_steps", "num_inference_steps", "inference_steps"):
    for obj in (model, model.config, getattr(model, "generation_config", None)):
        if obj is not None and hasattr(obj, attr):
            print(f"{type(obj).__name__}.{attr} =", getattr(obj, attr))
!grep -rn "inference_steps" /content/VibeVoice/demo/inference_from_file.py | head -5

VOICE = "/content/VibeVoice/demo/voices/en-Alice_woman.wav"
LAYERS = model.model.language_model.layers
print(len(LAYERS), "layers")

## Calibration — do NOT skip

One tiny generation with a raw recorder. This answers, empirically:
1. **calls_per_step** — does the CFG negative pass fire the layer hooks a second time per AR step? (hook calls ≈ 1 + steps, or 1 + 2×steps, or something else)
2. **turn boundaries** — does the emitted token stream contain per-turn speech_start/speech_end markers we can mask with?

In [ ]:
tiny_script = "Speaker 1: One short calibration sentence.\nSpeaker 1: And a second one to test turn boundaries.\n"
inputs = processor(
    text=[tiny_script], voice_samples=[[VOICE]], return_tensors="pt", padding=True
)
inputs = {k: (v.to("cuda") if hasattr(v, "to") else v) for k, v in inputs.items()}

rec = LayerActivationRecorder(LAYERS)  # raw: calls_per_step=1, we inspect counts
with rec, torch.inference_mode():
    out = model.generate(
        **inputs, tokenizer=processor.tokenizer, cfg_scale=1.3, max_new_tokens=None
    )

seq = out.sequences[0] if hasattr(out, "sequences") else out[0]
prompt_len = inputs["input_ids"].shape[1]
gen_ids = seq[prompt_len:].tolist()
n_gen = len(gen_ids)
print(f"hook calls: {rec.num_calls}   generated tokens: {n_gen}")
print(f"=> calls_per_step ≈ {(rec.num_calls - 1) / max(n_gen, 1):.2f} (prefill excluded)")

tok = processor.tokenizer
ids = {"start": tok.convert_tokens_to_ids("<|vision_start|>"),
       "end": tok.convert_tokens_to_ids("<|vision_end|>"),
       "frame": tok.convert_tokens_to_ids("<|vision_pad|>")}
print("special ids:", ids)
from collections import Counter
print("generated token histogram:", Counter(gen_ids).most_common(6))
starts = [i for i, t in enumerate(gen_ids) if t == ids["start"]]
ends = [i for i, t in enumerate(gen_ids) if t == ids["end"]]
print(f"speech_start positions: {starts}\nspeech_end positions: {ends}")
print("=> per-turn boundaries available!" if len(ends) >= 2 else "=> NO per-turn markers — use drop_first_fraction fallback")

**Fill these in from the output above before continuing:**

In [ ]:
CALLS_PER_STEP = 1     # <- from calibration (2 if CFG double-fires the hooks)
KEEP_CALL = 0          # <- if 2: verify which index is the positive pass
TURN_MASKING = False   # <- True if per-turn start/end markers were found
DROP_FIRST_FRACTION = 0.35  # fallback: lead-in is sentence 1 of 4, skip early frames
FRAME_ID = ids["frame"]; START_ID = ids["start"]; END_ID = ids["end"]

## Full capture loop — 80 generations

Lead-in sentence (pole-specific) + neutral target text (identical across poles), same voice, K=2 samples per condition. Vectors pooled over target-region speech frames only.

In [ ]:
import soundfile as sf, os, time
os.makedirs("/content/p0_audio", exist_ok=True)
K = 2
records, failures = [], []
t0 = time.time()

def target_mask(gen_ids):
    """Bool mask over generated steps: speech frames after the first turn ends."""
    m = torch.zeros(len(gen_ids), dtype=torch.bool)
    first_end = next((i for i, t in enumerate(gen_ids) if t == END_ID), None)
    for i, t in enumerate(gen_ids):
        if t == FRAME_ID and (first_end is None or i > first_end):
            m[i] = True
    return m

conditions = [(s, a, p) for s in CFG["scripts"] for a in CFG["axes"] for p in ("pos", "neg")]
for ci, (script_id, axis, pole) in enumerate(conditions):
    lead_in = CFG["axes"][axis][pole]
    text = f"Speaker 1: {lead_in}\nSpeaker 1: {CFG['scripts'][script_id]}\n"
    for k in range(K):
        try:
            inputs = processor(text=[text], voice_samples=[[VOICE]], return_tensors="pt", padding=True)
            inputs = {kk: (v.to("cuda") if hasattr(v, "to") else v) for kk, v in inputs.items()}
            rec = LayerActivationRecorder(LAYERS, calls_per_step=CALLS_PER_STEP, keep_call=KEEP_CALL)
            with rec, torch.inference_mode():
                out = model.generate(**inputs, tokenizer=processor.tokenizer, cfg_scale=1.3)
            seq = out.sequences[0] if hasattr(out, "sequences") else out[0]
            gen_ids = seq[inputs["input_ids"].shape[1]:].tolist()
            states = rec.step_states()
            mask = target_mask(gen_ids)[: states.shape[1]] if TURN_MASKING else None
            records.append(pool_record(
                states, script_id=script_id, axis=axis, pole=pole, sample_idx=k,
                num_calls_total=rec.num_calls, speech_frame_mask=mask,
                drop_first_fraction=0.0 if TURN_MASKING else DROP_FIRST_FRACTION,
            ))
            if hasattr(out, "speech_outputs") and out.speech_outputs and k == 0:
                wav = out.speech_outputs[0].detach().float().cpu().numpy().squeeze()
                sf.write(f"/content/p0_audio/{script_id}_{axis}_{pole}.wav", wav, 24000)
        except Exception as e:
            failures.append((script_id, axis, pole, k, repr(e)))
            print("FAIL:", script_id, axis, pole, k, repr(e)[:200])
    el = time.time() - t0
    print(f"[{ci+1}/{len(conditions)}] {script_id}/{axis}/{pole}  {len(records)} records  {el/60:.1f} min")

save_records(records, "/content/vectors.pt")
print(f"saved {len(records)} records, {len(failures)} failures")

## Honesty check (spec Stage 1.4) — listen to pole pairs

Same script, opposite poles. If you cannot HEAR a difference in affect, the extraction has no signal to find — note it and consider the Expresso fallback (`docs/resources.md` §4: high-arousal styles need Meta's improvised tarball).

In [ ]:
from IPython.display import Audio, display
for sid in list(CFG["scripts"])[:3]:
    for axis in CFG["axes"]:
        for pole in ("pos", "neg"):
            p = f"/content/p0_audio/{sid}_{axis}_{pole}.wav"
            print(sid, axis, pole)
            display(Audio(p))

## Save out (Colab disk is wiped!)

Download `/content/vectors.pt` and a few honesty-check wavs → put vectors.pt in `experiments/p0_steering/` locally (gitignored) and wavs in `experiments/p0_steering/audio/`. Record in NOTES.md: calibration findings (calls_per_step, turn markers yes/no), weights revision, solver step count, honesty-check listening verdict, failures count.

In [ ]:
from google.colab import files
files.download("/content/vectors.pt")